In [ ]:



import numpy as np
import pandas as pd
from scipy.stats import weibull_min, norm
from sklearn.cluster import KMeans



np.random.seed(42)

df = pd.read_csv(
    r"C:\Users\jiaqi\OneDrive\Desktop\produkt_ff_stunde_19570701_20241231_02667.txt",
    sep=";"
)

print("Raw data ")
print(df.head())

# Parse datetime
df["datetime"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d%H")
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month

print("\n=== Data with datetime features (head) ===")
print(df[["MESS_DATUM", "datetime", "date", "hour", "month"]].head())

# Handle column name for wind speed
# Prefer stripped "F", fallback to original "   F"
df_cols_stripped = {c.strip(): c for c in df.columns}
if "F" in df_cols_stripped:
    wind_col = df_cols_stripped["F"]
elif "   F" in df.columns:
    wind_col = "   F"
else:
    raise KeyError("Cannot find wind speed column 'F' or '   F' in the file.")


# select winter

winter = df["month"].isin([12, 1, 2])
df_w = df.loc[winter, ["date", "hour", wind_col]].copy()
df_w.rename(columns={wind_col: "F"}, inplace=True)

# clean
df_w["F"] = pd.to_numeric(df_w["F"], errors="coerce")
df_w = df_w[np.isfinite(df_w["F"])]
df_w = df_w[df_w["F"] >= 0]

print("\n=== Winter subset (head) ===")
print(df_w.head())


# Build daily 24-hour paths (avoid cross-day)

T = 24

# pivot: rows = date, columns = hour (0..23)
pivot = df_w.pivot_table(index="date", columns="hour", values="F", aggfunc="mean")

# keep only days with all 24 hours present
pivot = pivot.reindex(columns=list(range(24)))
pivot_complete = pivot.dropna(axis=0, how="any")

daily_paths = pivot_complete.to_numpy()  # shape (n_days, 24)

print("\n=== Daily 24h paths summary ===")
print("Number of complete winter days:", daily_paths.shape[0])
print("daily_paths shape:", daily_paths.shape)
print("Example day (first row, 24 hours):")
print(daily_paths[0])

if daily_paths.shape[0] < 50:
    print("\n[Warning] Complete days are not many. Copula estimation may be noisy.")


#  Weibull

all_wind = df_w["F"].to_numpy()
shape, loc, scale = weibull_min.fit(all_wind, floc=0)

print("\n=== Weibull parameters (fit on all winter hourly winds) ===")
print("shape:", shape, "loc:", loc, "scale:", scale)


#  Gaussian copula

eps = 1e-10
u_daily = weibull_min.cdf(daily_paths, shape, loc, scale)
u_daily = np.clip(u_daily, eps, 1 - eps)
z_daily = norm.ppf(u_daily)  # shape (n_days, 24)

# correlation matrix 24x24
corr = np.corrcoef(z_daily, rowvar=False)
corr = (corr + corr.T) / 2
corr += 1e-6 * np.eye(T)

print("\n=== Copula correlation matrix shape ===")
print(corr.shape)

# Monte Carlo s
n_mc = 1000
z_samples = np.random.multivariate_normal(
    mean=np.zeros(T),
    cov=corr,
    size=n_mc
)

u_samples = norm.cdf(z_samples)
wind_samples = weibull_min.ppf(u_samples, shape, loc, scale)  # shape (1000, 24)

print("\n=== Simulated wind samples shape ===")
print(wind_samples.shape)
print("First 2 samples (first 8 hours):")
print(wind_samples[:2, :8])


# KMeans -> 2 intra scenarios

kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
labels = kmeans.fit_predict(wind_samples)

intra_scenarios = kmeans.cluster_centers_  # (2, 24)
p_intra = np.array([np.mean(labels == i) for i in range(2)])  # (2,)

# day-ahead expectation
w_DA = np.sum(intra_scenarios * p_intra[:, None], axis=0)  # (24,)

print("\n=== Intra-day scenarios (wind speed, 10m) ===")
print("intra_scenarios shape:", intra_scenarios.shape)
print("First scenario (first 8 hours):", intra_scenarios[0, :8])
print("Second scenario (first 8 hours):", intra_scenarios[1, :8])

print("\n=== Intra-day scenario probabilities ===")
print(p_intra)

print("\n=== Day-ahead wind forecast (10m) ===")
print("w_DA shape:", w_DA.shape)
print("w_DA (first 8 hours):", w_DA[:8])


# build RT scenario tree (3 sigma per omega)

deltas = [-0.15, 0.02, 0.15]  # low/mid/high
scenario_tree = {}

for omega in range(2):
    base_path = intra_scenarios[omega]
    rt_paths = []
    for d in deltas:
        noise = np.random.normal(0, 0.05, T)
        path = base_path * (1 + d + noise)
        path = np.maximum(path, 0)
        rt_paths.append(path)
    scenario_tree[omega] = np.array(rt_paths)

# Wind speed -> hub height -> power

def adjust_to_hub_height(v, h_ref=10, h_hub=100, alpha=0.14):
    return v * (h_hub / h_ref) ** alpha

def power_curve(v, v_ci=3, v_r=12, v_co=25, P_r=3):
    if v < v_ci or v >= v_co:
        return 0.0
    elif v <= v_r:
        return P_r * ((v - v_ci) / (v_r - v_ci)) ** 3
    else:
        return P_r

vectorized_power = np.vectorize(power_curve)
n_turbines = 200

# RT at hub height + power
scenario_tree_hub_power = {}
for omega in scenario_tree:
    hub = adjust_to_hub_height(scenario_tree[omega])
    pw = vectorized_power(hub) * n_turbines
    scenario_tree_hub_power[omega] = pw

# intra at hub height + power
intra_scenarios_hub = adjust_to_hub_height(intra_scenarios)
intra_scenarios_hub_power = vectorized_power(intra_scenarios_hub) * n_turbines
# DA expected power
p_DA = np.sum(intra_scenarios_hub_power * p_intra[:, None], axis=0)

print("\n=== Intra-day power scenarios (wind farm) ===")
print("intra_scenarios_hub_power shape:", intra_scenarios_hub_power.shape)
print("Scenario 0 (first 8 hours):", intra_scenarios_hub_power[0, :8])
print("Scenario 1 (first 8 hours):", intra_scenarios_hub_power[1, :8])

print("\n=== Day-ahead power forecast (wind farm) ===")
print("p_DA shape:", p_DA.shape)
print("p_DA (first 8 hours):", p_DA[:8])

print("\n=== RT power scenario tree shapes ===")
for omega in scenario_tree_hub_power:
    print(f"omega={omega}, shape={scenario_tree_hub_power[omega].shape}")


print("\n\n--- FEED THESE TO PART 2 ---")
print("p_intra (len=2):", p_intra)
print("p_DA (len=24):", p_DA)
print("intra_scenarios_hub_power shape:", intra_scenarios_hub_power.shape)
print("scenario_tree_hub_power[0] shape:", scenario_tree_hub_power[0].shape)
print("scenario_tree_hub_power[1] shape:", scenario_tree_hub_power[1].shape)


print("\nFull p_DA:", p_DA)
print("\nFull intra_scenarios_hub_power:\n", intra_scenarios_hub_power)
print("\nFull scenario_tree_hub_power[0]:\n", scenario_tree_hub_power[0])
print("\nFull scenario_tree_hub_power[1]:\n", scenario_tree_hub_power[1])

In [3]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List

import pyomo.environ as pyo
from pyomo.opt import SolverFactory, TerminationCondition




@dataclass
class ModelData:
    T: List[int]
    G: List[int]
    R: List[int]
    N: List[int]
    B: List[int]
    OMEGA: List[int]
    SIGMA: List[int]
    G_n: Dict[int, List[int]]
    R_n: Dict[int, List[int]]
    B_n: Dict[int, List[int]]
    neighbors: Dict[int, List[int]]
    E0: Dict[int, float]
    pi_OMEGA: Dict[int, float]
    pi_SIGMA: Dict[Tuple[int, int], float]
    C_DA: Dict[int, float]
    C_ID_C: Dict[int, float]
    C_ID_R: Dict[int, float]
    C_RT: Dict[int, float]
    C_spill: float
    VOLL: float
    C_deg: Dict[int, float]
    Pmin: Dict[int, float]
    Pmax: Dict[int, float]
    RU: Dict[int, float]
    RD: Dict[int, float]
    W_DA: Dict[Tuple[int, int], float]
    W_ID: Dict[Tuple[int, int, int], float]
    W_RT: Dict[Tuple[int, int, int, int], float]
    D: Dict[Tuple[int, int], float]
    Fmax: Dict[Tuple[int, int], float]
    Bline: Dict[Tuple[int, int], float]
    eta_ch: Dict[int, float]
    eta_dis: Dict[int, float]
    Emin: Dict[int, float]
    Emax: Dict[int, float]
    delta_t: float


def build_model(data: ModelData) -> pyo.ConcreteModel:
    m = pyo.ConcreteModel()
    m.T = pyo.Set(initialize=data.T, ordered=True)
    m.G = pyo.Set(initialize=data.G)
    m.R = pyo.Set(initialize=data.R)
    m.N = pyo.Set(initialize=data.N)
    m.B = pyo.Set(initialize=data.B)
    m.OMEGA = pyo.Set(initialize=data.OMEGA)
    m.SIGMA = pyo.Set(initialize=data.SIGMA)
    T_sorted = list(data.T)
    T_first = T_sorted[0]


    ####### Variables ######

    #### First stage
    m.P = pyo.Var(m.G, m.T)
    m.W = pyo.Var(m.R, m.T, domain=pyo.NonNegativeReals)
    m.theta = pyo.Var(m.N, m.T)
    m.Pch = pyo.Var(m.B, m.T, domain=pyo.NonNegativeReals)
    m.Pdis = pyo.Var(m.B, m.T, domain=pyo.NonNegativeReals)
    m.E = pyo.Var(m.B, m.T)
    m.E_RT_exp = pyo.Var(m.B, m.T)

    #### Second stage
    m.dP_up = pyo.Var(m.G, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.dP_dn = pyo.Var(m.G, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.dW_up = pyo.Var(m.R, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.dW_dn = pyo.Var(m.R, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.dPch_ID = pyo.Var(m.B, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.dPdis_ID = pyo.Var(m.B, m.T, m.OMEGA, domain=pyo.NonNegativeReals)
    m.theta_ID = pyo.Var(m.N, m.T, m.OMEGA)
    m.E_ID = pyo.Var(m.B, m.T, m.OMEGA)

    #### Third stage
    m.rU = pyo.Var(m.G, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.rD = pyo.Var(m.G, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.spill = pyo.Var(m.R, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.shed = pyo.Var(m.N, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.dPch_RT = pyo.Var(m.B, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.dPdis_RT = pyo.Var(m.B, m.T, m.OMEGA, m.SIGMA, domain=pyo.NonNegativeReals)
    m.theta_RT = pyo.Var(m.N, m.T, m.OMEGA, m.SIGMA)
    m.E_RT = pyo.Var(m.B, m.T, m.OMEGA, m.SIGMA)


    ########## Objective ########
    def obj_rule(m):
        total = 0.0
        for omega in data.OMEGA:
            inner = 0.0
            for sigma in data.SIGMA:
                thermal = sum(
                    data.C_DA[i] * (
                        m.P[i, t]
                        + m.dP_up[i, t, omega] - m.dP_dn[i, t, omega]
                        + m.rU[i, t, omega, sigma] - m.rD[i, t, omega, sigma]
                    )
                    for i in data.G for t in data.T
                )
                spill_cost = sum(
                    data.C_spill * m.spill[q, t, omega, sigma]
                    for q in data.R for t in data.T
                )
                shed_cost = sum(
                    data.VOLL * m.shed[n, t, omega, sigma]
                    for n in data.N for t in data.T
                )

                inner += data.pi_SIGMA[(omega, sigma)] * (thermal + spill_cost + shed_cost)
            total += data.pi_OMEGA[omega] * inner
        return total

    m.Obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)
    ########################################
    ######### Constraints ##################
    ########################################

    ######### DA constraints ##################
    m.GenCap = pyo.Constraint(m.G, m.T,
        rule=lambda m, i, t: pyo.inequality(data.Pmin[i], m.P[i, t], data.Pmax[i]))

    def ramp_da(m, i, t):
        if t == T_first:
            return pyo.Constraint.Skip
        return pyo.inequality(-data.RD[i], m.P[i, t] - m.P[i, t - 1], data.RU[i])
    m.RampDA = pyo.Constraint(m.G, m.T, rule=ramp_da)

    m.WindDA = pyo.Constraint(m.R, m.T,
        rule=lambda m, q, t: m.W[q, t] <= data.W_DA[(q, t)])

    def balance_da(m, n, t):
        lhs = (
            sum(m.P[i, t] for i in data.G_n[n])
            + sum(m.W[q, t] for q in data.R_n[n])
            + sum(m.Pdis[b, t] - m.Pch[b, t] for b in data.B_n[n])
            - data.D[(n, t)]
        )
        rhs = sum(
            data.Bline[(n, r)] * (m.theta[n, t] - m.theta[r, t])
            for r in data.neighbors[n]
        )
        return lhs == rhs
    m.BalanceDA = pyo.Constraint(m.N, m.T, rule=balance_da)

    m.LineLimitDA = pyo.ConstraintList()
    for n in data.N:
        for r in data.neighbors[n]:
            for t in data.T:
                flow = data.Bline[(n, r)] * (m.theta[n, t] - m.theta[r, t])
                m.LineLimitDA.add(pyo.inequality(-data.Fmax[(n, r)], flow, data.Fmax[(n, r)]))

    # SoC in DA: first hour uses E0, later hours use previous expected RT SoC
    m.SocDAFirst = pyo.Constraint(m.B,
        rule=lambda m, b: m.E[b, T_first] ==
                          data.E0[b]
                          + data.eta_ch[b] * m.Pch[b, T_first] * data.delta_t
                          - (1 / data.eta_dis[b]) * m.Pdis[b, T_first] * data.delta_t)

    def soc_da_rest(m, b, t):
        if t == T_first:
            return pyo.Constraint.Skip
        return m.E[b, t] == (
            m.E_RT_exp[b, t - 1]
            + data.eta_ch[b] * m.Pch[b, t] * data.delta_t
            - (1 / data.eta_dis[b]) * m.Pdis[b, t] * data.delta_t
        )
    m.SocDARest = pyo.Constraint(m.B, m.T, rule=soc_da_rest)

    m.SocCapDA = pyo.Constraint(m.B, m.T,
        rule=lambda m, b, t: pyo.inequality(data.Emin[b], m.E[b, t], data.Emax[b]))

    m.RefAngleDA = pyo.Constraint(m.T, rule=lambda m, t: m.theta[1, t] == 0)

    ########### ID constraints   ###########

    def balance_id(m, n, t, omega):
        lhs = (
            sum(m.dP_up[i, t, omega] - m.dP_dn[i, t, omega] for i in data.G_n[n])
            + sum(m.dW_up[q, t, omega] - m.dW_dn[q, t, omega] for q in data.R_n[n])
            + sum(m.dPdis_ID[b, t, omega] - m.dPch_ID[b, t, omega] for b in data.B_n[n])
        )
        rhs = sum(
            data.Bline[(n, r)] *
            ((m.theta_ID[n, t, omega] - m.theta_ID[r, t, omega]) - (m.theta[n, t] - m.theta[r, t]))
            for r in data.neighbors[n]
        )
        return lhs == rhs
    m.BalanceID = pyo.Constraint(m.N, m.T, m.OMEGA, rule=balance_id)

    m.ConvFeasID = pyo.Constraint(
        m.G, m.T, m.OMEGA,
        rule=lambda m, i, t, omega: pyo.inequality(
            data.Pmin[i],
            m.P[i, t] + m.dP_up[i, t, omega] - m.dP_dn[i, t, omega],
            data.Pmax[i]
        )
    )

    def ramp_id(m, i, t, omega):
        if t == T_first:
            return pyo.Constraint.Skip
        expr_t = m.P[i, t] + m.dP_up[i, t, omega] - m.dP_dn[i, t, omega]
        expr_tm1 = m.P[i, t - 1] + m.dP_up[i, t - 1, omega] - m.dP_dn[i, t - 1, omega]
        return pyo.inequality(-data.RD[i], expr_t - expr_tm1, data.RU[i])
    m.RampID = pyo.Constraint(m.G, m.T, m.OMEGA, rule=ramp_id)

    m.WindID = pyo.Constraint(
        m.R, m.T, m.OMEGA,
        rule=lambda m, q, t, omega: pyo.inequality(
            0.0,
            m.W[q, t] + m.dW_up[q, t, omega] - m.dW_dn[q, t, omega],
            data.W_ID[(q, t, omega)]
        )
    )

    m.SocID = pyo.Constraint(
        m.B, m.T, m.OMEGA,
        rule=lambda m, b, t, omega:
            m.E_ID[b, t, omega] ==
            m.E[b, t]
            + data.eta_ch[b] * m.dPch_ID[b, t, omega] * data.delta_t
            - (1 / data.eta_dis[b]) * m.dPdis_ID[b, t, omega] * data.delta_t
    )

    m.SocCapID = pyo.Constraint(
        m.B, m.T, m.OMEGA,
        rule=lambda m, b, t, omega:
            pyo.inequality(data.Emin[b], m.E_ID[b, t, omega], data.Emax[b])
    )

    m.LineLimitID = pyo.ConstraintList()
    for n in data.N:
        for r in data.neighbors[n]:
            for t in data.T:
                for omega in data.OMEGA:
                    flow = data.Bline[(n, r)] * (m.theta_ID[n, t, omega] - m.theta_ID[r, t, omega])
                    m.LineLimitID.add(pyo.inequality(-data.Fmax[(n, r)], flow, data.Fmax[(n, r)]))

    m.RefAngleID = pyo.Constraint(m.T, m.OMEGA,
        rule=lambda m, t, omega: m.theta_ID[1, t, omega] == 0)


    ###################### RT constraints   ###########

    def balance_rt(m, n, t, omega, sigma):
        lhs = (
            sum(m.rU[i, t, omega, sigma] - m.rD[i, t, omega, sigma] for i in data.G_n[n])
            + sum(
                data.W_RT[(q, t, omega, sigma)]
                - (m.W[q, t] + m.dW_up[q, t, omega] - m.dW_dn[q, t, omega])
                - m.spill[q, t, omega, sigma]
                for q in data.R_n[n]
            )
            + sum(m.dPdis_RT[b, t, omega, sigma] - m.dPch_RT[b, t, omega, sigma] for b in data.B_n[n])
            + m.shed[n, t, omega, sigma]
        )
        rhs = sum(
            data.Bline[(n, r)] *
            ((m.theta_RT[n, t, omega, sigma] - m.theta_RT[r, t, omega, sigma]) -
             (m.theta_ID[n, t, omega] - m.theta_ID[r, t, omega]))
            for r in data.neighbors[n]
        )
        return lhs == rhs
    m.BalanceRT = pyo.Constraint(m.N, m.T, m.OMEGA, m.SIGMA, rule=balance_rt)

    m.ConvFeasRT = pyo.Constraint(
        m.G, m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, i, t, omega, sigma: pyo.inequality(
            data.Pmin[i],
            m.P[i, t]
            + m.dP_up[i, t, omega] - m.dP_dn[i, t, omega]
            + m.rU[i, t, omega, sigma] - m.rD[i, t, omega, sigma],
            data.Pmax[i]
        )
    )

    def ramp_rt(m, i, t, omega, sigma):
        if t == T_first:
            return pyo.Constraint.Skip
        expr_t = (m.P[i, t]
                  + m.dP_up[i, t, omega] - m.dP_dn[i, t, omega]
                  + m.rU[i, t, omega, sigma] - m.rD[i, t, omega, sigma])
        expr_tm1 = (m.P[i, t - 1]
                    + m.dP_up[i, t - 1, omega] - m.dP_dn[i, t - 1, omega]
                    + m.rU[i, t - 1, omega, sigma] - m.rD[i, t - 1, omega, sigma])
        return pyo.inequality(-data.RD[i], expr_t - expr_tm1, data.RU[i])
    m.RampRT = pyo.Constraint(m.G, m.T, m.OMEGA, m.SIGMA, rule=ramp_rt)

    m.ReserveU = pyo.Constraint(
        m.G, m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, i, t, omega, sigma: m.rU[i, t, omega, sigma] <= data.RU[i])
    m.ReserveD = pyo.Constraint(
        m.G, m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, i, t, omega, sigma: m.rD[i, t, omega, sigma] <= data.RD[i])

    m.SocRT = pyo.Constraint(
        m.B, m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, b, t, omega, sigma:
            m.E_RT[b, t, omega, sigma] ==
            m.E_ID[b, t, omega]
            + data.eta_ch[b] * m.dPch_RT[b, t, omega, sigma] * data.delta_t
            - (1 / data.eta_dis[b]) * m.dPdis_RT[b, t, omega, sigma] * data.delta_t
    )

    m.SocCapRT = pyo.Constraint(
        m.B, m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, b, t, omega, sigma:
            pyo.inequality(data.Emin[b], m.E_RT[b, t, omega, sigma], data.Emax[b])
    )

    m.LineLimitRT = pyo.ConstraintList()
    for n in data.N:
        for r in data.neighbors[n]:
            for t in data.T:
                for omega in data.OMEGA:
                    for sigma in data.SIGMA:
                        flow = data.Bline[(n, r)] * (m.theta_RT[n, t, omega, sigma] - m.theta_RT[r, t, omega, sigma])
                        m.LineLimitRT.add(pyo.inequality(-data.Fmax[(n, r)], flow, data.Fmax[(n, r)]))

    m.RefAngleRT = pyo.Constraint(
        m.T, m.OMEGA, m.SIGMA,
        rule=lambda m, t, omega, sigma: m.theta_RT[1, t, omega, sigma] == 0
    )

    # Expected SoC
    def soc_exp(m, b, t):
        return m.E_RT_exp[b, t] == sum(
            data.pi_OMEGA[omega] * sum(
                data.pi_SIGMA[(omega, sigma)] * m.E_RT[b, t, omega, sigma]
                for sigma in data.SIGMA
            )
            for omega in data.OMEGA
        )
    m.SocExpected = pyo.Constraint(m.B, m.T, rule=soc_exp)
    m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)
    return m


def df_2d(var, rows, cols, row_name="i", col_name="t"):
    arr = np.zeros((len(rows), len(cols)))
    for ri, r in enumerate(rows):
        for ci, c in enumerate(cols):
            arr[ri, ci] = pyo.value(var[r, c])
    out = pd.DataFrame(arr, index=list(rows), columns=list(cols))
    out.index.name = row_name
    out.columns.name = col_name
    return out


######### DATA (basically the same as 3 hour version, but  load are  extended)


T = list(range(1, 25))
G  = [1, 2, 3, 4]
R  = [1]
N  = [1, 2, 3, 4, 5]
B  = [1]
OMEGA = [1, 2]
SIGMA = [1, 2, 3]

######## scenario probabilities result is a bit of different from 3hour version
pi_OMEGA = {1:  0.431, 2:0.569}
pi_SIGMA = {(1,1): 1/5, (1,2): 3/5, (1,3): 1/5,
            (2,1): 1/5, (2,2): 3/5, (2,3): 1/5}


####### Wind parameters, results are form scenarios.py


p_DA = np.array([
    21.19324334, 23.19226145, 24.21549516, 22.08088694, 22.49638444, 25.34419118,
    25.8652865,  22.53768716, 22.87365236, 22.04320584, 23.90681905, 24.08508236,
    23.7885077,  26.029808,   24.82700357, 22.43283157, 20.76805055, 21.80124863,
    18.90287587, 19.39308452, 18.62244501, 19.56938158, 18.46373337, 17.05830265
], dtype=float)

intra_scenarios_hub_power = np.array([
    [1.55118586e-01, 1.48828915e-01, 1.79706125e-01, 1.47430645e-01,
     1.39809762e-01, 9.02352027e-02, 1.63861943e-01, 6.20741047e-02,
     5.57645025e-02, 5.29120624e-02, 4.39052056e-02, 6.79132083e-02,
     7.09991407e-02, 5.80863381e-02, 8.23503650e-02, 9.71428972e-02,
     1.38039858e-01, 1.26752945e-01, 1.88459515e-01, 2.39438389e-01,
     2.13294936e-01, 3.44017400e-01, 3.21875830e-01, 3.99282215e-01],
    [4.89674730e+01, 5.36138696e+01, 5.59471981e+01, 5.10371204e+01,
     5.20112127e+01, 5.86841006e+01, 5.97959375e+01, 5.22096682e+01,
     5.29974997e+01, 5.10744753e+01, 5.54102946e+01, 5.57922036e+01,
     5.51000213e+01, 6.03173013e+01, 5.74945388e+01, 5.19200864e+01,
     4.80034939e+01, 5.04156060e+01, 4.36093791e+01, 4.46794526e+01,
     4.29259401e+01, 4.49504308e+01, 4.24143527e+01, 3.90513018e+01]
], dtype=float)

scenario_tree_hub_power_0 = np.array([
    [0.00000000e+00, 0.00000000e+00, 2.66082171e-02, 3.88380343e-02,
     0.00000000e+00, 0.00000000e+00, 1.30251683e-02, 6.33985159e-11,
     1.34332222e-06, 6.52490537e-05, 0.00000000e+00, 0.00000000e+00,
     0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 5.04076432e-03,
     0.00000000e+00, 2.84846343e-05, 1.01807291e-03, 1.87607646e-04,
     8.18392095e-03, 1.61409571e-04, 6.10730630e-02, 3.13384103e-02],
    [8.95728408e-02, 6.82531588e-01, 2.87761078e-01, 1.06732750e-01,
     4.55745077e-01, 2.09966681e-01, 3.18006272e-02, 5.30703541e-02,
     1.17997797e-01, 8.58616227e-04, 7.92436021e-02, 1.23365182e-01,
     2.38186583e-02, 1.20949528e-01, 1.25908015e-01, 3.60265901e-02,
     3.99349213e-01, 3.54030143e-01, 2.07466847e-01, 2.81692652e-01,
     4.92014362e-01, 3.98194936e-01, 7.28580551e-02, 7.13796287e-01],
    [7.37409882e-01, 9.01484452e-01, 2.28301323e+00, 1.02073242e+00,
     1.05768952e+00, 1.06308651e+00, 1.46553038e+00, 3.88447973e-01,
     6.17261833e-01, 6.34306552e-01, 6.59530610e-01, 8.23878188e-01,
     2.27497337e+00, 3.13082112e-01, 1.26369143e+00, 1.49686242e+00,
     1.32396583e+00, 7.08878032e-01, 2.53088060e+00, 4.38341116e-01,
     9.77265053e-01, 2.94429301e+00, 1.50163328e+00, 3.73638596e+00]
], dtype=float)

scenario_tree_hub_power_1 = np.array([
    [ 10.63263028,  14.12692377,  18.66960569,  23.57329177,  19.21650171,
      20.06139566,  38.59837153,  34.35193778,  18.2954165 ,  10.2813766 ,
       9.31762502,  24.53042802,  27.81164447,  25.23072843,  22.792559  ,
      15.36627699,  23.85196195,  34.78157024,   6.52228557,  16.97067493,
      11.36299558,   9.60980476,  10.45623167,  17.17059546],
    [ 60.18552888,  84.19868602,  62.11303064,  34.85351997,  54.25623397,
      44.95561019,  48.19936131,  66.85437116,  61.69289269,  65.56115751,
      56.02640524,  74.93462679,  54.90762665,  87.07749823,  69.32396062,
      54.60082503,  56.25232535,  55.67794044,  71.56480204,  52.88946281,
      47.71890618,  46.92882528,  58.36377806,  52.16187533],
    [ 93.60137637,  83.89691745,  92.39427645,  99.08579168, 137.05262257,
     111.06823351, 105.08879017,  71.12885358, 143.74598304, 115.78872931,
     130.29233762, 135.39548334, 106.71648721, 122.0397811 , 153.71342979,
     108.57539729, 109.93994894, 121.52523156, 102.05031633,  88.81510184,
      87.38738548, 107.72815128, 102.28303081, 102.87905487]
], dtype=float)

omega_to_part1_idx = {1: 0, 2: 1}


W_DA = {(1, t): float(np.round(p_DA[t-1], 4)) for t in T}

W_ID = {}
for omega in OMEGA:
    idx = omega_to_part1_idx[omega]
    for t in T:
        W_ID[(1, t, omega)] = float(np.round(intra_scenarios_hub_power[idx, t-1], 4))

W_RT = {}
for omega in OMEGA:
    idx = omega_to_part1_idx[omega]
    rt_mat = scenario_tree_hub_power_1 if idx == 1 else scenario_tree_hub_power_0
    for sigma in SIGMA:
        for t in T:
            W_RT[(1, t, omega, sigma)] = float(np.round(rt_mat[sigma-1, t-1], 4))

# Costs (same as 3 hour)

C_DA = {1: 30.0, 2: 25.0, 3: 40.0, 4: 70.0}
C_ID_C = {1: 5.0, 2: 5.0, 3: 2.0, 4: 10.0}
C_ID_R = {1: 5.0}
C_RT   = {1: 5.0, 2: 5.0, 3: 2.0, 4: 10.0}
C_spill = 5.0
VOLL    = 1000.0
C_deg   = {1: 1.0}

Pmin = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
Pmax = {1: 100.0, 2: 80.0, 3: 70.0, 4: 60.0}
RU   = {1: 20.0, 2: 20.0, 3: 20.0, 4: 20.0}
RD   = {1: 20.0, 2: 20.0, 3: 20.0, 4: 20.0}



######## Realistic 24-hour LOAD

TotalLoad_24 = {
     1: 215,  2: 210,  3: 205,  4: 203,  5: 205,  6: 215,
     7: 235,  8: 255,  9: 275, 10: 290, 11: 300, 12: 305,
    13: 300, 14: 295, 15: 290, 16: 295, 17: 310, 18: 320,
    19: 315, 20: 300, 21: 285, 22: 270, 23: 250, 24: 230
}

node_share = {
    1: 110 / (110 + 30 + 86.7 + 76.7),
    2:  30 / (110 + 30 + 86.7 + 76.7),
    3: 86.7 / (110 + 30 + 86.7 + 76.7),
    4: 76.7 / (110 + 30 + 86.7 + 76.7),
    5: 0.0
}



D = {}
for t in range(1, 25):
    for n in [1,2,3,4,5]:
        D[(n,t)] = TotalLoad_24[t] * node_share[n]

neighbors = {
    1: [2,4],
    2: [1,3,5],
    3: [2],
    4: [1,5],
    5: [2,4]
}

edges = [(1,2),(1,4),(2,1),(2,3),(2,5),(3,2),(4,1),(4,5),(5,2),(5,4)]
Fmax = {(n,r): 120.0 for (n,r) in edges}
Bline = {(n,r): 6.67  for (n,r) in edges}

G_n = {1:[1,2], 2:[], 3:[3], 4:[4], 5:[]}
R_n = {1:[], 2:[], 3:[], 4:[], 5:[1]}
B_n = {1:[], 2:[1], 3:[], 4:[], 5:[]}

eta_ch = {1: 0.95}
eta_dis = {1: 0.95}
Emin = {1: 0.0}
Emax = {1: 100.0}
E0   = {1: 50.0}
delta_t = 1.0

data = ModelData(
    T=T, G=G, R=R, N=N, B=B, OMEGA=OMEGA, SIGMA=SIGMA,
    G_n=G_n, R_n=R_n, B_n=B_n, neighbors=neighbors,
    E0=E0,
    pi_OMEGA=pi_OMEGA, pi_SIGMA=pi_SIGMA,
    C_DA=C_DA, C_ID_C=C_ID_C, C_ID_R=C_ID_R, C_RT=C_RT,
    C_spill=C_spill, VOLL=VOLL, C_deg=C_deg,
    Pmin=Pmin, Pmax=Pmax, RU=RU, RD=RD,
    W_DA=W_DA, W_ID=W_ID, W_RT=W_RT,
    D=D,
    Fmax=Fmax, Bline=Bline,
    eta_ch=eta_ch, eta_dis=eta_dis, Emin=Emin, Emax=Emax,
    delta_t=delta_t
)

########SOLVE
model = build_model(data)
solver = SolverFactory("highs")
results = solver.solve(model, tee=False)

status = results.solver.termination_condition
print("\nSolver Status:", status)

if status in (TerminationCondition.optimal, TerminationCondition.locallyOptimal):
    total_cost = pyo.value(model.Obj)
    print(f"Total Expected Cost: ${total_cost:.2f}")
else:
    print("Careful: Model did not solve to optimality!")




# #####Max load shedding

max_shed = max(
    pyo.value(model.shed[n, t, omega, sigma])
    for n in data.N
    for t in data.T
    for omega in data.OMEGA
    for sigma in data.SIGMA
)
print("Max shed:", max_shed)


# #######Day-ahead nodal prices

prices = []
n_price = 1

for t in data.T:
    lam = pyo.value(model.dual[model.BalanceDA[n_price, t]])
    prices.append(-float(lam))





########### OUTPUTS

print("\nDay-Ahead Generator Dispatch (P) ")
print(df_2d(model.P, data.G, data.T, row_name="i", col_name="t"))

print("\nDay-Ahead Battery SoC (E)")
print(df_2d(model.E, data.B, data.T, row_name="b", col_name="t"))

print("\n End battery (Expected RT SoC)")
print(df_2d(model.E_RT_exp, data.B, data.T, row_name="b", col_name="t"))








sum_w_id = {omega: sum(W_ID[(1,t,omega)] for t in T) for omega in OMEGA}
print(sum_w_id)
# Table : expected dual of RT SoC constraints  =====
def build_table3_expected_duals(model, data, use_negative=True):
    rows = []
    b = data.B[0]  #

    for t in data.T:
        row = {"t": t}
        for omega in data.OMEGA:
            val = 0.0
            for sigma in data.SIGMA:
                c = model.SocRT[b, t, omega, sigma]
                d = model.dual.get(c, 0.0)
                if use_negative:
                    d = -d
                val += data.pi_SIGMA[(omega, sigma)] * float(d)
            row[f"omega{omega}"] = val
        rows.append(row)

    df = pd.DataFrame(rows)

    df = df.rename(columns={"omega1": "σ1 (low)", "omega2": "σ2 (high)"})
    return df

df_table3 = build_table3_expected_duals(model, data, use_negative=True)
print("\nTable (Expected dual values of RT SoC constraints):")
print(df_table3.round(3))


Solver Status: optimal
Total Expected Cost: $172230.16
Max shed: 0.0

Day-Ahead Generator Dispatch (P) 
t        1         2         3         4         5        6        7   \
i                                                                       
1  100.0000  100.0000  100.0000  100.0000  100.0000  97.1935  96.6724   
2   80.0000   66.0232   80.0000   78.4155   60.0000  80.0000  80.0000   
3   13.8068   20.7845    0.7845    2.5036   22.5036  12.4623  32.4623   
4   -0.0000   -0.0000   -0.0000   -0.0000   -0.0000  -0.0000  -0.0000   

t        8         9         10  ...       15        16        17        18  \
i                                ...                                          
1  100.0000  100.0000  100.0000  ...  100.000  100.0000  100.0000  100.0000   
2   80.0000   80.0000   80.0000  ...   80.000   80.0000   80.0000   80.0000   
3   52.4623   70.0000   70.0000  ...   70.000   70.0000   70.0000   70.0000   
4   -0.0000    2.1263   17.9568  ...   15.173   22.5672   39.

In [4]:

#  result in TABLE


def build_template_like_cost_table(model, data):
    rows = []
    for t in data.T:
        unit_costs = {}
        for i in data.G:
            exp_cost_it = 0.0
            for omega in data.OMEGA:
                for sigma in data.SIGMA:
                    prob = data.pi_OMEGA[omega] * data.pi_SIGMA[(omega, sigma)]
                    realized = (
                        pyo.value(model.P[i, t])
                        + pyo.value(model.dP_up[i, t, omega]) - pyo.value(model.dP_dn[i, t, omega])
                        + pyo.value(model.rU[i, t, omega, sigma]) - pyo.value(model.rD[i, t, omega, sigma])
                    )
                    exp_cost_it += prob * data.C_DA[i] * realized
            unit_costs[i] = exp_cost_it
        exp_spill_t = 0.0
        exp_shed_t = 0.0
        for omega in data.OMEGA:
            for sigma in data.SIGMA:
                prob = data.pi_OMEGA[omega] * data.pi_SIGMA[(omega, sigma)]
                exp_spill_t += prob * sum(
                    data.C_spill * pyo.value(model.spill[q, t, omega, sigma])
                    for q in data.R
                )
                exp_shed_t += prob * sum(
                    data.VOLL * pyo.value(model.shed[n, t, omega, sigma])
                    for n in data.N
                )

        total_t = sum(unit_costs.values()) + exp_spill_t + exp_shed_t
        row = {"t": t}
        for i in data.G:
            row[f"Unit {i}"] = unit_costs[i]
        row["spill"] = exp_spill_t
        row["shed"] = exp_shed_t
        row["total"] = total_t
        rows.append(row)

    df = pd.DataFrame(rows)
    total_row = {"t": "total"}
    for i in data.G:
        total_row[f"Unit {i}"] = df[f"Unit {i}"].sum()
    total_row["spill"] = df["spill"].sum()
    total_row["shed"] = df["shed"].sum()
    total_row["total"] = df["total"].sum()
    df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)

    return df



df_cost = build_template_like_cost_table(model, data)

print("Cost table (24h)")

with pd.option_context("display.max_rows", 30, "display.max_columns", 20):
    print(df_cost.round(2))

print("\nCheck totals:")
print("Objective from Pyomo =", round(pyo.value(model.Obj), 6))
print("Total from table     =", round(float(df_cost.loc[df_cost['t']=="total", "total"].iloc[0]), 6))


Cost table (24h)
        t    Unit 1    Unit 2    Unit 3   Unit 4  spill  shed      total
0       1   2979.62   2000.00      0.00     0.00    0.0   0.0    4979.62
1       2   3000.00   2000.00      0.00     0.00    0.0   0.0    5000.00
2       3   2937.70   2000.00      0.00     0.00    0.0   0.0    4937.70
3       4   2917.38   1975.62     59.37     0.00    0.0   0.0    4952.37
4       5   2863.44   1918.72    430.50     0.00    0.0   0.0    5212.66
5       6   2868.76   1975.62    667.41     0.00    0.0   0.0    5511.79
6       7   2905.15   2000.00   1194.68     0.00    0.0   0.0    6099.83
7       8   2973.43   2000.00   1900.29     0.00    0.0   0.0    6873.72
8       9   2893.81   1943.10   2109.29     0.00    0.0   0.0    6946.20
9      10   2950.76   2000.00   2560.61     0.00    0.0   0.0    7511.37
10     11   2979.98   2000.00   2617.92     0.00    0.0   0.0    7597.90
11     12   2989.69   2000.00   2708.96   348.23    0.0   0.0    8046.88
12     13   3000.00   2000.00   28

In [5]:
def rt_lmp(model, data, n=2):  # battery in node2
    out = {}
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            out[(omega, sigma)] = []
            for t in data.T:
                lam = pyo.value(model.dual[model.BalanceRT[n, t, omega, sigma]])
                out[(omega, sigma)].append(-float(lam))
    return out

for omega in data.OMEGA:
    for sigma in data.SIGMA:
        s = sum(pyo.value(model.spill[q,t,omega,sigma]) for q in data.R for t in data.T)
        print("omega",omega,"sigma",sigma,"total spill",s)

for k,con in enumerate(model.LineLimitRT.values(), start=1):
    d = model.dual.get(con, 0.0)
    if abs(d) > 1e-6:
        print("binding LineLimitRT idx", k, "dual", d)


for omega in data.OMEGA:
    for sigma in data.SIGMA:
        for t in data.T:

            e = pyo.value(model.E_RT[1,t,omega,sigma])
            if e < data.Emin[1] + 1e-4 or e > data.Emax[1] - 1e-4:
                print("hit bound at t",t,"omega",omega,"sigma",sigma,"E_RT",e)



def rt_lmp_node(model, data, n=2):
    out = {}
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            out[(omega, sigma)] = []
            for t in data.T:
                lam = pyo.value(model.dual[model.BalanceRT[n, t, omega, sigma]])
                out[(omega, sigma)].append(-float(lam))
                return out

lmp2 = rt_lmp_node(model, data, n=2)
for k,v in lmp2.items():
    print(k, "min/mean/max", min(v), sum(v)/len(v), max(v))

def realized_dispatch(model, data, i, t, omega, sigma):
    return (pyo.value(model.P[i,t])
            + pyo.value(model.dP_up[i,t,omega]) - pyo.value(model.dP_dn[i,t,omega])
            + pyo.value(model.rU[i,t,omega,sigma]) - pyo.value(model.rD[i,t,omega,sigma]))

for omega in data.OMEGA:
    for sigma in data.SIGMA:
        cnt = 0
        for t in data.T:
            p4 = realized_dispatch(model, data, 4, t, omega, sigma)
            if p4 > 1e-3:
                cnt += 1
        print("omega",omega,"sigma",sigma,"hours Unit4 on:",cnt)



def soc_expected_paths(model, data, b=1):
    rows = []
    for t in data.T:
        row = {"t": t, "E_RT_exp": pyo.value(model.E_RT_exp[b, t])}
        for omega in data.OMEGA:
            row[f"E_RT_cond_omega{omega}"] = sum(
                data.pi_SIGMA[(omega, sigma)] * pyo.value(model.E_RT[b, t, omega, sigma])
                for sigma in data.SIGMA
            )
        rows.append(row)
    return pd.DataFrame(rows)

df_soc = soc_expected_paths(model, data, b=1)
print("\n=== Expected RT SoC paths ===")
print(df_soc.round(4))

# 1) ID wind total by omega
sum_w_id = {omega: sum(W_ID[(1,t,omega)] for t in T) for omega in OMEGA}
print("sum_w_id:", sum_w_id)

# 2) Expected RT wind total by omega (conditional on omega)
sum_w_rt_cond = {}
for omega in OMEGA:
    s = 0.0
    for t in T:
        s += sum(data.pi_SIGMA[(omega,sigma)] * W_RT[(1,t,omega,sigma)] for sigma in SIGMA)
    sum_w_rt_cond[omega] = s
print("sum_E[W_RT|omega]:", sum_w_rt_cond)


omega 1 sigma 1 total spill 0.0
omega 1 sigma 2 total spill 0.0
omega 1 sigma 3 total spill 0.0
omega 2 sigma 1 total spill 0.0
omega 2 sigma 2 total spill 0.0
omega 2 sigma 3 total spill 0.0
hit bound at t 12 omega 1 sigma 1 E_RT -0.0
hit bound at t 13 omega 1 sigma 1 E_RT -0.0
hit bound at t 14 omega 1 sigma 1 E_RT -0.0
hit bound at t 19 omega 1 sigma 1 E_RT -0.0
hit bound at t 20 omega 1 sigma 1 E_RT -0.0
hit bound at t 21 omega 1 sigma 1 E_RT -0.0
hit bound at t 22 omega 1 sigma 1 E_RT -0.0
hit bound at t 23 omega 1 sigma 1 E_RT -0.0
hit bound at t 24 omega 1 sigma 1 E_RT -0.0
hit bound at t 12 omega 1 sigma 2 E_RT -0.0
hit bound at t 13 omega 1 sigma 2 E_RT -0.0
hit bound at t 14 omega 1 sigma 2 E_RT -0.0
hit bound at t 19 omega 1 sigma 2 E_RT -0.0
hit bound at t 20 omega 1 sigma 2 E_RT -0.0
hit bound at t 21 omega 1 sigma 2 E_RT -0.0
hit bound at t 22 omega 1 sigma 2 E_RT -0.0
hit bound at t 23 omega 1 sigma 2 E_RT -0.0
hit bound at t 24 omega 1 sigma 2 E_RT -0.0
hit bound at t 1

In [6]:
import pandas as pd
import pyomo.environ as pyo


def realized_dispatch(model, i, t, omega, sigma):
    """Realized thermal output of unit i at hour t in (omega,sigma)."""
    return (pyo.value(model.P[i, t])
            + pyo.value(model.dP_up[i, t, omega]) - pyo.value(model.dP_dn[i, t, omega])
            + pyo.value(model.rU[i, t, omega, sigma]) - pyo.value(model.rD[i, t, omega, sigma]))


### Cost decomposition

def table2_cost_decomposition(model, data):
    rows = []

    # Generator cost by unit i
    gen_cost = {}
    for i in data.G:
        c = 0.0
        for omega in data.OMEGA:
            for sigma in data.SIGMA:
                prob = data.pi_OMEGA[omega] * data.pi_SIGMA[(omega, sigma)]
                for t in data.T:
                    c += prob * data.C_DA[i] * realized_dispatch(model, i, t, omega, sigma)
        gen_cost[i] = c

    ### Spill cost
    spill_cost = 0.0
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            prob = data.pi_OMEGA[omega] * data.pi_SIGMA[(omega, sigma)]
            for q in data.R:
                for t in data.T:
                    spill_cost += prob * data.C_spill * pyo.value(model.spill[q, t, omega, sigma])

    # ##Shed cost
    shed_cost = 0.0
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            prob = data.pi_OMEGA[omega] * data.pi_SIGMA[(omega, sigma)]
            for n in data.N:
                for t in data.T:
                    shed_cost += prob * data.VOLL * pyo.value(model.shed[n, t, omega, sigma])

    ##Build table
    row = {"period": "total_24h"}
    for i in data.G:
        row[f"Unit{i}"] = gen_cost[i]
    row["spill"] = spill_cost
    row["shed"] = shed_cost
    row["total"] = sum(gen_cost.values()) + spill_cost + shed_cost
    rows.append(row)

    df = pd.DataFrame(rows).set_index("period")
    return df

df_cost = table2_cost_decomposition(model, data)
print("\n=== Table 2 (24h): Cost decomposition ===")
print(df_cost.round(2))


df_cost_share = (df_cost / df_cost["total"].iloc[0]) * 100
print("\n=== Cost share (%) ===")
print(df_cost_share.round(2))



##ID/RT dispatch adjustment summaries

def summarize_ID_adjustments(model, data):
    """ID redispatch summary by omega and unit (sum over time)."""
    records = []
    for omega in data.OMEGA:
        for i in data.G:
            up = sum(pyo.value(model.dP_up[i, t, omega]) for t in data.T)
            dn = sum(pyo.value(model.dP_dn[i, t, omega]) for t in data.T)
            net = up - dn
            records.append({
                "omega": omega,
                "unit": i,
                "ID_sum_dP_up": up,
                "ID_sum_dP_dn": dn,
                "ID_net_(up-dn)": net,
                "ID_abs_(up+dn)": up + dn
            })
    return pd.DataFrame(records)

def summarize_RT_regulation(model, data):
    """RT regulation summary by (omega,sigma) and unit (sum over time)."""
    records = []
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            for i in data.G:
                up = sum(pyo.value(model.rU[i, t, omega, sigma]) for t in data.T)
                dn = sum(pyo.value(model.rD[i, t, omega, sigma]) for t in data.T)
                net = up - dn
                records.append({
                    "omega": omega,
                    "sigma": sigma,
                    "unit": i,
                    "RT_sum_rU": up,
                    "RT_sum_rD": dn,
                    "RT_net_(rU-rD)": net,
                    "RT_abs_(rU+rD)": up + dn
                })
    return pd.DataFrame(records)

def summarize_battery_energy(model, data):
    b = data.B[0]
    dt = data.delta_t

    # DA totals
    da_ch = sum(pyo.value(model.Pch[b, t]) for t in data.T) * dt
    da_dis = sum(pyo.value(model.Pdis[b, t]) for t in data.T) * dt

    # ID totals by omega
    id_rows = []
    for omega in data.OMEGA:
        id_ch = sum(pyo.value(model.dPch_ID[b, t, omega]) for t in data.T) * dt
        id_dis = sum(pyo.value(model.dPdis_ID[b, t, omega]) for t in data.T) * dt
        id_rows.append({"omega": omega, "ID_charge": id_ch, "ID_discharge": id_dis, "ID_net(dis-ch)": id_dis - id_ch})
    ## RT totals by (omega,sigma)
    rt_rows = []
    for omega in data.OMEGA:
        for sigma in data.SIGMA:
            rt_ch = sum(pyo.value(model.dPch_RT[b, t, omega, sigma]) for t in data.T) * dt
            rt_dis = sum(pyo.value(model.dPdis_RT[b, t, omega, sigma]) for t in data.T) * dt
            rt_rows.append({
                "omega": omega, "sigma": sigma,
                "RT_charge": rt_ch, "RT_discharge": rt_dis, "RT_net(dis-ch)": rt_dis - rt_ch
            })

    out = {
        "DA_totals": pd.DataFrame([{"DA_charge": da_ch, "DA_discharge": da_dis, "DA_net(dis-ch)": da_dis - da_ch}]),
        "ID_totals_by_omega": pd.DataFrame(id_rows),
        "RT_totals_by_path": pd.DataFrame(rt_rows),
    }
    return out

df_id = summarize_ID_adjustments(model, data)
print("\n=== ID adjustments summary (by omega, unit) ===")
print(df_id.round(3))

df_rt = summarize_RT_regulation(model, data)
print("\n=== RT regulation summary (by omega, sigma, unit) ===")
print(df_rt.round(3))

bat = summarize_battery_energy(model, data)
print("\n=== Battery DA totals (energy) ===")
print(bat["DA_totals"].round(3))
print("\n=== Battery ID totals by omega (energy) ===")
print(bat["ID_totals_by_omega"].round(3))
print("\n=== Battery RT totals by (omega,sigma) (energy) ===")
print(bat["RT_totals_by_path"].round(3))



def key_hours_RT(model, data, omega, sigma, K=6):
    """Return top K hours with largest total RT regulation |rU|+|rD| across units."""
    scores = []
    for t in data.T:
        tot = sum(pyo.value(model.rU[i,t,omega,sigma]) + pyo.value(model.rD[i,t,omega,sigma]) for i in data.G)
        scores.append((t, tot))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:K]

def key_hours_ID(model, data, omega, K=6):
    """Return top K hours with largest total ID adjustment (dP_up+dP_dn) across units."""
    scores = []
    for t in data.T:
        tot = sum(pyo.value(model.dP_up[i,t,omega]) + pyo.value(model.dP_dn[i,t,omega]) for i in data.G)
        scores.append((t, tot))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:K]

print("\n=== Key hours (ID) ===")
for omega in data.OMEGA:
    print("omega", omega, "top hours:", key_hours_ID(model, data, omega, K=6))

print("\n=== Key hours (RT) ===")
for omega in data.OMEGA:
    for sigma in data.SIGMA:
        print("omega", omega, "sigma", sigma, "top hours:", key_hours_RT(model, data, omega, sigma, K=6))




=== Table 2 (24h): Cost decomposition ===
              Unit1     Unit2     Unit3    Unit4  spill  shed      total
period                                                                  
total_24h  70826.81  47748.49  43785.83  9869.03    0.0   0.0  172230.16

=== Cost share (%) ===
           Unit1  Unit2  Unit3  Unit4  spill  shed  total
period                                                   
total_24h  41.12  27.72  25.42   5.73    0.0   0.0  100.0

=== ID adjustments summary (by omega, unit) ===
   omega  unit  ID_sum_dP_up  ID_sum_dP_dn  ID_net_(up-dn)  ID_abs_(up+dn)
0      1     1         3.328        34.490         -31.162          37.818
1      1     2         5.561       137.560        -131.999         143.121
2      1     3       151.271         0.785         150.487         152.056
3      1     4       437.637         6.093         431.544         443.731
4      2     1         0.000        72.682         -72.682          72.682
5      2     2        15.406        25.64